### TOOLs

moddels can request to call tools that performs tasks - such as fetching data from a database, searching the web, or running the code. - tools are pairing of : 

1. A schema , including name of tool, a description and / or arguments, definitions often in JSON schema
2. A function or coroutine to execute

In [17]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")

model = init_chat_model(
    model="openai/gpt-4o",
    model_provider="openrouter",
    temperature=0.7,
    max_tokens=500,
    streaming=True,
)
response = model.invoke("Write me a 200 words paragraph on Artificial Intelligence")
response.text

"Artificial Intelligence (AI) represents a transformative force in the modern world, reshaping industries, enhancing efficiencies, and redefining human interaction with technology. At its core, AI involves the development of computer systems capable of performing tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation. The rapid advancements in machine learning, a subset of AI, have enabled computers to learn from data, identify patterns, and make informed decisions with minimal human intervention. This capability is being leveraged across various sectors, including healthcare, finance, transportation, and entertainment, leading to innovations like personalized medicine, autonomous vehicles, and sophisticated recommendation systems. AI's potential for revolutionizing the future is vast, promising improvements in productivity and the creation of new economic opportunities. However, it also poses significan

In [18]:
from langchain.tools import tool

@tool
def get_weather(location: str)-> str:
    """to get weather info man!!"""
    return f"weather in {location}is sunny"


model_with_tools=model.bind_tools([get_weather])

In [24]:
from langchain_core.messages import ToolMessage

response = model_with_tools.invoke("Howdy modi - kesa hai weather delhi ka?")
print(response)

tool_messages = []
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

    tool_result = get_weather.invoke(tool_call["args"])
    print(f"Result: {tool_result}")
    tool_messages.append(
        ToolMessage(content=tool_result, tool_call_id=tool_call["id"])
    )

final_response = model_with_tools.invoke([response, *tool_messages])
print(final_response.content)

content='' additional_kwargs={} response_metadata={'finish_reason': 'tool_calls', 'model_name': 'openai/gpt-4o', 'system_fingerprint': 'fp_38255cd5f6', 'id': 'gen-1790370116-wVOEeI2Pc6n7ddp7OyKE', 'created': 1790370116, 'object': 'chat.completion.chunk', 'model_provider': 'openrouter', 'cost': 0.000275, 'cost_details': {'upstream_inference_completions_cost': 0.00014, 'upstream_inference_prompt_cost': 0.000135, 'upstream_inference_cost': 0.000275}} id='lc_run--01a0da5f-f292-7003-ac88-c566d1318524' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Delhi'}, 'id': 'call_5D7mxdBNuDxTs8iVokXrwXSx', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 54, 'output_tokens': 14, 'total_tokens': 68, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}}
Tool: get_weather
Args: {'location': 'Delhi'}
Result: weather in Delhiis sunny
The weather in Delhi is currently sunny.


In [25]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is currently sunny.


In [26]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'openai/gpt-4o', 'system_fingerprint': 'fp_38255cd5f6', 'id': 'gen-1790370874-85OcgF600DjqmtJkU4XD', 'created': 1790370874, 'object': 'chat.completion.chunk', 'model_provider': 'openrouter', 'cost': 0.0002625, 'cost_details': {'upstream_inference_completions_cost': 0.00014, 'upstream_inference_prompt_cost': 0.0001225, 'upstream_inference_cost': 0.0002625}}, id='lc_run--01a0da6b-83a5-73c0-b24c-74d5bff0b7bf', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_9jc5KdwQmCnZTJFh30NsRy0A', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 14, 'total_tokens': 63, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}}),
 ToolMessage(content='weather in Bostonis sunny', name='get_weather', tool_call_